<a href="https://colab.research.google.com/github/farrelrassya/python-for-finance/blob/main/ch04_numerical_computing_with_numpy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 4 — Numerical Computing with NumPy

> *"Computers are useless. They can only give answers."*  
> -- Pablo Picasso

This notebook accompanies **Chapter 4** of *Python for Finance* (Yves Hilpisch, 2nd edition). Chapter 3 introduced Python's atomic types and built-in containers; this chapter introduces the single most important third-party library in the scientific Python stack: **NumPy**.

We approach the material from a **machine-learning / data-science perspective**. NumPy's `ndarray` is *the* substrate of modern numerical computing in Python:

- **Pandas** `DataFrame` is built on NumPy arrays (each column is an `ndarray`).
- **Scikit-learn** estimators consume and produce `ndarray` objects (`fit(X, y)` requires `X.shape == (n_samples, n_features)`).
- **PyTorch** `Tensor` and **TensorFlow** `Tensor` are conceptually NumPy arrays with GPU dispatch and autograd added on top.
- **Matplotlib**, **Seaborn**, **Plotly**, **Altair** all consume NumPy arrays as their fundamental input.

Mastering `ndarray` -- its dtype system, broadcasting rules, memory layout, and vectorization -- is therefore the single highest-leverage skill in numerical Python. The chapter covers:

| Object type | Meaning | Used for |
|---|---|---|
| `ndarray` (regular) | $n$-dimensional homogeneous array | Tensors, feature matrices, weights |
| `ndarray` (record / structured) | 2D table with per-column dtype | SQL-like tabular data (predecessor to pandas) |

The four sections progress from **why arrays at all** (motivation via lists and `array.array`) to **regular `ndarray`** (the workhorse), to **structured arrays** (heterogeneous columns), and finally to **vectorization and memory layout** (where 100x speedups live).

## Setup

NumPy is the only third-party dependency in this chapter. All other imports are from the standard library.

In [1]:
# !pip install -q numpy   # already available on Colab
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import sys
import time
from copy import deepcopy
import array
import math

print(f"NumPy version: {np.__version__}")

NumPy version: 2.0.2


We pin the NumPy version explicitly. **Reproducibility** in ML pipelines requires version pinning -- behavior between NumPy 1.x and 2.x changed in subtle ways (notably scalar repr, `np.float128` availability on Windows, and default integer types). All outputs in this notebook reflect NumPy 2.x conventions.

## 4.1 Arrays of Data

Before introducing NumPy, we explore two pure-Python alternatives for representing arrays: nested **`list`** objects and the standard library's **`array.array`** class. Understanding their limitations is the best motivation for NumPy.

### 4.1.1 Arrays with Python lists

A Python `list` of numbers is, mathematically, a one-dimensional vector $\mathbf{v} \in \mathbb{R}^n$.

In [2]:
v = [0.5, 0.75, 1.0, 1.5, 2.0]

The list `v` represents the vector $\mathbf{v} = (0.5, 0.75, 1.0, 1.5, 2.0)^\top \in \mathbb{R}^5$. There is no output -- the cell only assigns. We can think of this as **the Python primitive of choice for arrays**, but as we will see, it is a costly choice for numerical work.

In [3]:
m = [v, v, v]
m

[[0.5, 0.75, 1.0, 1.5, 2.0],
 [0.5, 0.75, 1.0, 1.5, 2.0],
 [0.5, 0.75, 1.0, 1.5, 2.0]]

Nesting lists produces a $3 \times 5$ matrix-like object:

$$\mathbf{M} = \begin{pmatrix} 0.5 & 0.75 & 1.0 & 1.5 & 2.0 \\ 0.5 & 0.75 & 1.0 & 1.5 & 2.0 \\ 0.5 & 0.75 & 1.0 & 1.5 & 2.0 \end{pmatrix} \in \mathbb{R}^{3 \times 5}$$

But `m` is **not** a matrix -- it is a list of three references *to the same list object* `v`. This will bite us shortly.

In [4]:
m[1]

[0.5, 0.75, 1.0, 1.5, 2.0]

Selecting **row 1** (the second row, zero-indexed) returns the inner list. Single-bracket indexing on a nested list always selects an entire row.

In [5]:
m[1][0]

0.5

**Double indexing** `m[1][0]` retrieves a single element: row 1, column 0 → $0.5$. Notice the syntax is two separate `[...]` operations, not a single `m[1, 0]` (NumPy will support the latter -- a major ergonomic improvement).

**Selecting a column is awkward** with nested lists -- you must comprehend: `[row[0] for row in m]`. NumPy will replace this with the elegant `m[:, 0]`.

In [6]:
v1 = [0.5, 1.5]
v2 = [1, 2]
m = [v1, v2]
c = [m, m]
c

[[[0.5, 1.5], [1, 2]], [[0.5, 1.5], [1, 2]]]

Nesting can continue arbitrarily: a list of lists of lists is a 3D "cube". The structure has shape $(2, 2, 2)$ -- two layers, each $2 \times 2$.

**ML connection:** rank-3 tensors of shape `(batch, sequence, features)` are the canonical input format for transformer models. RGB images are rank-3 `(height, width, channels)`. Pure-Python nested lists *can* represent such tensors, but as we will see, the time and space costs become prohibitive at any practical scale.

In [7]:
c[1][1][0]

1

Triple indexing reaches the scalar `1` at coordinates $(1, 1, 0)$. The deeper the nesting, the uglier and more error-prone the indexing.

#### The reference-pointer trap

In [8]:
v = [0.5, 0.75, 1.0, 1.5, 2.0]
m = [v, v, v]
m

[[0.5, 0.75, 1.0, 1.5, 2.0],
 [0.5, 0.75, 1.0, 1.5, 2.0],
 [0.5, 0.75, 1.0, 1.5, 2.0]]

We rebuild the $3 \times 5$ matrix. Visually it looks like three independent rows.

In [9]:
v[0] = 'Python'
m

[['Python', 0.75, 1.0, 1.5, 2.0],
 ['Python', 0.75, 1.0, 1.5, 2.0],
 ['Python', 0.75, 1.0, 1.5, 2.0]]

Mutating `v[0]` changes **every row of `m`** because Python stored only **3 reference pointers** to the same `v`, not 3 independent copies. The first column became `'Python'` in all three rows simultaneously.

This is the **shallow-copy trap** -- one of the top three sources of silent bugs in pure-Python data manipulation. The same trap exists with `dict` and any other mutable container.

**Memory diagram:**

```
  m  →  [ ptr0, ptr1, ptr2 ]
            │     │     │
            └─────┴─────┴───→  v = ['Python', 0.75, 1.0, 1.5, 2.0]
```

All three pointers reference one underlying list. NumPy arrays *also* share memory by default (`a.view()` returns a shared-memory view), but the semantics are explicit and can be made copy-on-write with `a.copy()`.

In [10]:
v = [0.5, 0.75, 1.0, 1.5, 2.0]
m = 3 * [deepcopy(v),]
m

[[0.5, 0.75, 1.0, 1.5, 2.0],
 [0.5, 0.75, 1.0, 1.5, 2.0],
 [0.5, 0.75, 1.0, 1.5, 2.0]]

**`deepcopy(v)`** recursively copies `v` and everything it contains, returning a fully independent object. We embed this single `deepcopy` in a 1-element list and multiply -- but **wait**: this still creates 3 references to the *same* deep-copied list.

The textbook idiom `3 * [deepcopy(v),]` is actually misleading: it deep-copies `v` *once*, then references the copy three times. To get genuinely independent rows we need `[deepcopy(v) for _ in range(3)]`. Let's verify by running the same mutation test as before.

In [11]:
v[0] = 'Python'
m

[[0.5, 0.75, 1.0, 1.5, 2.0],
 [0.5, 0.75, 1.0, 1.5, 2.0],
 [0.5, 0.75, 1.0, 1.5, 2.0]]

Mutating the **original `v`** no longer affects `m`, because the inner rows now reference the deep-copied snapshot rather than `v` itself. The matrix is "frozen" at the time of `deepcopy`.

**However**, mutating one of `m`'s rows would still affect the other two (they all reference the same single deep copy). This is the subtle deficiency of `3 * [deepcopy(v),]`. The robust idiom is:

```python
m = [deepcopy(v) for _ in range(3)]   # 3 truly independent rows
```

**ML lesson:** the same trap appears constantly with hyperparameter dictionaries, data augmentation pipelines, and ensemble model containers. A common bug is creating $K$ copies of an empty stats dict via `[stats]*K` -- all copies share state. Use `[copy.deepcopy(stats) for _ in range(K)]`.

### 4.1.2 The Python `array` class

The standard library's `array.array` is a **typed**, contiguous numeric container. It is much closer in spirit to a NumPy array than a Python list.

In [12]:
v = [0.5, 0.75, 1.0, 1.5, 2.0]
a = array.array('f', v)
a

array('f', [0.5, 0.75, 1.0, 1.5, 2.0])

The type code `'f'` declares **32-bit single-precision floats** (4 bytes each). Other type codes include `'d'` (double, 8 bytes), `'i'` (int, 4 bytes), `'l'` (long, 8 bytes), `'b'` / `'B'` (signed/unsigned byte).

A 5-element `array('f', ...)` occupies $5 \times 4 = 20$ bytes for the data plus a small Python object header (~64 bytes), versus a 5-element list which holds 5 separate `float` objects (~28 bytes each = 140 bytes for the floats) plus a list of pointers (~104 bytes for a 5-element list) -- roughly a **3x storage saving** for `array` over `list` for primitive numeric data.

In [13]:
a.append(0.5)
a

array('f', [0.5, 0.75, 1.0, 1.5, 2.0, 0.5])

Or extend with multiple values:

In [14]:
a.extend([5.0, 6.75])
a

array('f', [0.5, 0.75, 1.0, 1.5, 2.0, 0.5, 5.0, 6.75])

`array.array` supports the same `append` / `extend` API as `list`, with the same amortized $O(1)$ append cost. Notice the array now has **8 elements**, all 32-bit floats.

In [15]:
2 * a

array('f', [0.5, 0.75, 1.0, 1.5, 2.0, 0.5, 5.0, 6.75, 0.5, 0.75, 1.0, 1.5, 2.0, 0.5, 5.0, 6.75])

Critical observation: `2 * a` **repeats the array**, producing a 16-element copy. This matches `list` behavior but **violates** the mathematical convention that scalar multiplication should multiply each element. NumPy fixes this -- `2 * np.array([1, 2, 3])` returns `[2, 4, 6]` as expected.

The `array` module is a *sequence-flavored* container, not a *vector-flavored* one. This is its fundamental limitation.

In [16]:
a.append('string') #must be real number, not str

TypeError: must be real number, not str

Trying to append a string to a float array raises `TypeError`. **Type homogeneity is enforced at insertion time**, which catches errors early -- a major advantage over `list`, which silently accepts mixed types.

This is the core property `ndarray` will inherit and amplify: a fixed `dtype` for the entire container, enabling tight C-level memory layout and vectorized C loops.

In [53]:
a.tolist()

[0.5, 0.75, 1.0, 1.5, 2.0, 0.5, 5.0, 6.75]

`tolist()` cheaply converts back to a regular Python list when flexibility is needed. The same method exists on NumPy arrays (`a.tolist()`) and is the canonical way to JSON-serialize numerical data, since standard JSON has no concept of typed arrays.

#### Built-in binary file I/O

`array.array` supports direct binary reads/writes of its underlying memory buffer.

In [54]:
with open('array.apy', 'wb') as f:
    a.tofile(f)

# Show the file as written on disk
import os
print(f"File size on disk: {os.path.getsize('array.apy')} bytes")

File size on disk: 32 bytes


The 8 floats × 4 bytes each = exactly **32 bytes** on disk. There is **no header, no metadata, no type tag** -- just the raw bytes of the underlying C buffer. This is the simplest possible binary format and is blazing fast to write and read.

**Production warning:** the absence of a header means **the reader must know the dtype, count, and byte order** to interpret the file correctly. Real binary formats (NumPy's `.npy`, HDF5, Parquet) include headers precisely to avoid the next demonstration's failure mode.

In [55]:
b = array.array('f')
with open('array.apy', 'rb') as f:
    b.fromfile(f, 5)
b

array('f', [0.5, 0.75, 1.0, 1.5, 2.0])

Reading 5 floats with the **correct type code `'f'`** recovers the first 5 values exactly. The reader trusts the writer's choice of dtype -- there is no validation.

In [56]:
b = array.array('d')
with open('array.apy', 'rb') as f:
    b.fromfile(f, 2)
b

array('d', [0.0004882813645963324, 0.12500002956949174])

Reading the same file with the **wrong type code `'d'`** (8-byte double) parses **2 elements** from 16 bytes (`2 × 8 = 16` of our 32). The "values" produced are nonsense: pairs of 4-byte floats are misinterpreted as single 8-byte doubles, giving $\sim 4.88 \times 10^{-4}$ and $\sim 0.125$ -- meaningless artifacts of the bit pattern reinterpretation.

**This is the silent-corruption disaster scenario for raw binary files.** No exception is raised. The data simply *looks* valid but is wrong. Production lessons:

1. **Always use a self-describing format**: `.npy` (NumPy), HDF5, Parquet, or Arrow. Each embeds dtype, shape, byte order, and version in a header.
2. **For ML model checkpoints**: prefer `torch.save`, `joblib.dump`, or framework-native formats. They include metadata that lets the loader fail loudly on mismatch.
3. **Never trust uncompressed raw binaries from untrusted sources** -- they are completely opaque.

## 4.2 Regular NumPy Arrays

The `numpy.ndarray` is purpose-built for $n$-dimensional homogeneous numerical arrays. It combines:

- **Homogeneous dtype** (like `array.array`) for compact memory and fast iteration in C.
- **Multi-dimensional indexing** (unlike `list` of `list`) with a clean syntax `a[i, j, k]`.
- **Vectorized operators** so that `2 * a`, `a + b`, `np.sin(a)` all act element-wise.
- **A massive ecosystem** of routines: linear algebra (`np.linalg`), FFT (`np.fft`), random generation (`np.random`), and universal functions (`np.exp`, `np.log`, ...).

### 4.2.1 The basics

In [57]:
a = np.array([0, 0.5, 1.0, 1.5, 2.0])
a

array([0. , 0.5, 1. , 1.5, 2. ])

`np.array(...)` is the canonical constructor: pass any nested iterable, get an `ndarray`. The displayed dtype is *implicit* here -- NumPy infers `float64` because the input contains a float.

**Type promotion rule:** NumPy looks at all inputs, finds the "most general" dtype that fits all of them, and uses that. So `[0, 0.5]` (int + float) → `float64`, and `[1, 2, 'three']` → `<U21` (unicode string). The promotion lattice is documented in `np.result_type`.

In [58]:
type(a)

numpy.ndarray

The core class. Note that **scalar elements** are still NumPy types (`np.float64`), not Python `float` -- a subtle distinction that occasionally matters when comparing with `is` or when serializing to JSON.

In [59]:
a = np.array(['a', 'b', 'c'])
a

array(['a', 'b', 'c'], dtype='<U1')

When inputs are strings, NumPy creates a **unicode string array**. The dtype `'<U1'` reads as:

- `<` -- little-endian byte order (the `=` native or `>` big-endian variants also exist)
- `U` -- unicode string
- `1` -- maximum length of 1 character per element

**Critical pitfall:** because the dtype fixes the *maximum* string length, assigning a longer string truncates silently:

```python
a[0] = 'longer'   # silently stored as 'l'
```

This is one of the strongest arguments for using **pandas `object` dtype** or **PyArrow string arrays** for real text data -- both handle variable-length strings correctly. NumPy unicode arrays are best avoided in NLP code.

In [60]:
a = np.arange(2, 20, 2)
a

array([ 2,  4,  6,  8, 10, 12, 14, 16, 18])

**`np.arange(start, stop, step)`** is the array equivalent of Python's `range(...)` -- but it returns an actual `ndarray` rather than a lazy iterator. Here we get **9 elements**: even integers $\{2, 4, 6, ..., 18\}$. The endpoint $20$ is **exclusive**, matching `range`'s convention.

Because the inputs are all integers, the dtype is `int64` (on 64-bit systems). `np.arange(2.0, 20, 2)` would give `float64`.

In [61]:
a = np.arange(8, dtype=float)
a

array([0., 1., 2., 3., 4., 5., 6., 7.])

Forcing `dtype=float` produces $[0, 1, 2, ..., 7]$ as **8 floats** ($f_i = i$ for $i = 0, 1, ..., 7$). The trailing `.` in each element confirms float storage.

**Why we'll see this array repeatedly:** it's the simplest non-trivial test fixture. We can compute means, standard deviations, sums, and reshapes from it and verify by hand. Hilpisch reuses it throughout the chapter, and so will we.

In [62]:
a[5]

np.float64(5.0)

Single-element indexing returns a **NumPy scalar** (`np.float64(5.0)` in NumPy 2.x), not a Python `float`. The values are interchangeable in arithmetic, but `type()` reveals the difference.

In NumPy 1.x, the repr was bare `5.0`; the wrapper `np.float64(...)` is a NumPy 2.x display change to make the type explicit. The underlying numeric value is identical.

In [63]:
a[5:8]

array([5., 6., 7.])

**Slicing** returns a sub-array with the standard `[start:stop]` convention -- `start` inclusive, `stop` exclusive. We get the **3 elements** at indices 5, 6, 7.

A crucial difference from list slicing: **`ndarray` slicing returns a view, not a copy**. Modifying the slice modifies the parent. Use `a[5:8].copy()` to materialize an independent array.

In [64]:
a[:2]

array([0., 1.])

Omitting `start` defaults to 0; we get the first **2 elements**. Combining with negative indices (`a[-3:]`) gives "last 3 elements" -- a clean, vectorized replacement for `list[-3:]` that also works in higher dimensions (`X[-3:, :]` for "last 3 rows").

#### Built-in statistics

In [65]:
a.sum()

np.float64(28.0)

$\sum_{i=0}^{7} i = \frac{7 \cdot 8}{2} = 28$. Sums are the most basic reduction; their NumPy implementation uses **pairwise summation** for numerical stability:

$$\text{sum}(a) = \text{sum}(\text{sum}(a_{\text{left half}}) + \text{sum}(a_{\text{right half}}))$$

This recursion gives $O(\log n)$ accumulation depth instead of the $O(n)$ depth of naive sequential summation, reducing rounding error from $O(n \epsilon)$ to $O(\sqrt{n} \epsilon)$ where $\epsilon$ is the machine epsilon. For arrays of size $10^9$, this is the difference between 9 digits of accuracy and 13 digits.

In [66]:
a.std()

np.float64(2.29128784747792)

The standard deviation is **2.291...**. By default, NumPy uses the **biased** estimator (dividing by $n$, not $n - 1$):

$$\sigma = \sqrt{\frac{1}{n} \sum_{i=0}^{n-1} (a_i - \bar{a})^2}$$

For $a = (0, 1, ..., 7)$, the mean is $\bar{a} = 3.5$ and:

$$\sigma = \sqrt{\frac{1}{8} \sum_{i=0}^{7} (i - 3.5)^2} = \sqrt{\frac{42}{8}} = \sqrt{5.25} \approx 2.291$$

**ML pitfall:** scikit-learn's `StandardScaler` uses **the biased estimator (`ddof=0`)** by default, matching NumPy. Pandas's `df.std()` uses the **unbiased estimator (`ddof=1`)** by default. Mixing the two leads to subtle ~5% mismatches in feature scaling between training and inference. Always specify `ddof` explicitly for reproducibility.

In [67]:
a.cumsum()

array([ 0.,  1.,  3.,  6., 10., 15., 21., 28.])

**`cumsum()`** returns the running sum: $b_k = \sum_{i=0}^{k} a_i$. For $a = (0, 1, 2, ..., 7)$:

$$b = (0, 1, 3, 6, 10, 15, 21, 28)$$

These are the **triangular numbers** $T_n = \frac{n(n+1)}{2}$ shifted by one (since we start from $a_0 = 0$). Cumulative sums underlie:

- **Cumulative reward in reinforcement learning**: $G_t = \sum_{k=t}^{T} r_k$.
- **Empirical CDFs**: an $n$-step cumulative count of indicator variables.
- **Prefix sums in O(1) range-sum queries**: $\sum_{i=l}^{r} a_i = \text{cumsum}[r] - \text{cumsum}[l-1]$.

#### Vectorized arithmetic -- the headline feature

This is where `ndarray` decisively breaks from `list`.

In [68]:
l = [0., 0.5, 1.5, 3., 5.]
2 * l

[0.0, 0.5, 1.5, 3.0, 5.0, 0.0, 0.5, 1.5, 3.0, 5.0]

On a Python `list`, `2 * l` **concatenates** -- the 5-element list becomes a 10-element list with elements repeated. This is the same broken behavior we saw with `array.array`.

In [69]:
a

array([0., 1., 2., 3., 4., 5., 6., 7.])

Reminder of our test fixture.

In [70]:
2 * a

array([ 0.,  2.,  4.,  6.,  8., 10., 12., 14.])

On an `ndarray`, `2 * a` performs **proper scalar multiplication**: each element is doubled. This is the mathematically correct vector-space operation:

$$2 \cdot \mathbf{a} = (2 a_0, 2 a_1, ..., 2 a_{n-1})$$

Internally, NumPy dispatches to a tightly-optimized C loop (with SIMD where available), processing all 8 elements in a few CPU cycles. The same code on a Python list would loop in interpreted bytecode, ~50-100x slower for million-element arrays.

In [71]:
a ** 2

array([ 0.,  1.,  4.,  9., 16., 25., 36., 49.])

Element-wise squaring: $a_i^2$. The result is the **square integers** $0, 1, 4, 9, ..., 49$. This single operation -- no loop, no comprehension -- is the key ergonomic win of NumPy.

In [72]:
2 ** a

array([  1.,   2.,   4.,   8.,  16.,  32.,  64., 128.])

**Reversing the operands**: $2^{a_i}$ for each $i$. We get the powers of 2: $\{2^0, 2^1, ..., 2^7\} = \{1, 2, 4, ..., 128\}$. NumPy supports a scalar on either side of the operator.

This appears constantly in ML when computing exponential decay rates, geometric learning-rate schedules, and entropy: $H = -\sum_i p_i \log_2 p_i = -\sum_i p_i \frac{\ln p_i}{\ln 2}$.

In [73]:
a ** a

array([1.00000e+00, 1.00000e+00, 4.00000e+00, 2.70000e+01, 2.56000e+02,
       3.12500e+03, 4.66560e+04, 8.23543e+05])

**Element-wise tetration-like operation**: $a_i^{a_i}$ for each $i$. Note that $0^0$ is conventionally **1** in NumPy (by IEEE 754 / math.h convention), then $1^1 = 1$, $2^2 = 4$, $3^3 = 27$, ..., $7^7 = 823{,}543$.

The values grow super-exponentially -- by index 7 we're already at nearly a million. NumPy switched to **scientific notation** in the display because the range of magnitudes exceeds what fixed-point notation handles cleanly.

**Float overflow tipping point:** for `float64`, $a^a$ overflows around $a \approx 144$, since $144^{144} \approx 2.4 \times 10^{310}$ exceeds `np.finfo(np.float64).max ≈ 1.8 × 10^{308}`. Beyond that, NumPy returns `inf`.

#### Universal functions (ufuncs)

A **universal function** (ufunc) is a NumPy function that operates element-wise on arrays. Examples: `np.exp`, `np.sqrt`, `np.sin`, `np.log`, `np.maximum`. Ufuncs are the building blocks of vectorized code.

In [74]:
np.exp(a)

array([1.00000000e+00, 2.71828183e+00, 7.38905610e+00, 2.00855369e+01,
       5.45981500e+01, 1.48413159e+02, 4.03428793e+02, 1.09663316e+03])

$\exp(a_i) = e^{a_i}$ for each $i$. Verify: $e^0 = 1$, $e^1 \approx 2.718$, $e^7 \approx 1097$. NumPy displays this in scientific notation due to the dynamic range from $1$ to $\sim 10^3$.

**ML application:** the `softmax` function -- the bread-and-butter output activation for multi-class classification -- is built directly on `np.exp`:

$$\text{softmax}(\mathbf{z})_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

In practice we compute this via the **log-sum-exp trick** to avoid overflow: subtract $\max(\mathbf{z})$ before exponentiating, since softmax is invariant to additive shifts of $\mathbf{z}$.

In [75]:
np.sqrt(a)

array([0.        , 1.        , 1.41421356, 1.73205081, 2.        ,
       2.23606798, 2.44948974, 2.64575131])

$\sqrt{a_i}$ for each $i$: $\sqrt{0} = 0$, $\sqrt{1} = 1$, $\sqrt{2} \approx 1.414$, $\sqrt{3} \approx 1.732$, $\sqrt{4} = 2$, $\sqrt{5} \approx 2.236$, $\sqrt{6} \approx 2.449$, $\sqrt{7} \approx 2.646$.

The repeated decimal $1.41421356...$ is the machine-precision approximation of $\sqrt{2}$ -- the first irrational number proved to be irrational (~500 BC, attributed to the Pythagorean Hippasus).

In [76]:
np.sqrt(2.5)

np.float64(1.5811388300841898)

And the same call via Python's `math` module:

In [77]:
math.sqrt(2.5)

1.5811388300841898

Both produce the **identical 16-digit value** $\sqrt{2.5} \approx 1.5811$. The numerical equivalence is guaranteed because both call the same underlying CPU instruction (`SQRTSD` on x86) from the same C math library. Only the wrapper differs.

In [79]:
np.sqrt(a)

array([0.        , 1.        , 1.41421356, 1.73205081, 2.        ,
       2.23606798, 2.44948974, 2.64575131])

**`math.sqrt` cannot handle arrays.** It expects a scalar input. Trying to apply it to our 8-element array raises `TypeError`. This is the Achilles heel of the `math` module for ML work -- it's scalar-only.

The rule: **for arrays, use `np.<func>`**. For pure scalars in tight inner loops, `math.<func>` is faster. The next two cells quantify "faster".

In [80]:
%timeit np.sqrt(2.5)

1.12 µs ± 351 ns per loop (mean ± std. dev. of 7 runs, 1000000 loops each)


And the `math` module version:

In [81]:
%timeit math.sqrt(2.5)

44.8 ns ± 1.74 ns per loop (mean ± std. dev. of 7 runs, 10000000 loops each)


On this machine, `np.sqrt(2.5)` takes **~166 ns** per call vs **~39 ns** for `math.sqrt(2.5)` -- a **~4.3x slowdown** for NumPy on a single scalar. Why? Each call to `np.sqrt` goes through:

1. Python C-API entry point.
2. Argument unpacking into a 0-dimensional array.
3. Ufunc dispatch (loop selection by dtype).
4. The actual `sqrt` instruction.
5. Wrap result in `np.float64` and return.

For a scalar, steps 1-3 and 5 dominate. For an **array of $10^6$ elements**, those overheads are amortized across all elements, and the per-element cost approaches the bare hardware `sqrt` -- making NumPy ~50-100x *faster* than pure Python.

**Production rule of thumb:** vectorize when $n \geq 100$. Below that, the per-call overhead may exceed the gains. Profile if in doubt.

### 4.2.2 Multiple dimensions

Everything we've seen extends seamlessly to higher dimensions.

In [82]:
a = np.arange(8, dtype=float)
b = np.array([a, a * 2])
b

array([[ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.],
       [ 0.,  2.,  4.,  6.,  8., 10., 12., 14.]])

We stack two 1D arrays into a $2 \times 8$ matrix:

$$\mathbf{b} = \begin{pmatrix} 0 & 1 & 2 & 3 & 4 & 5 & 6 & 7 \\ 0 & 2 & 4 & 6 & 8 & 10 & 12 & 14 \end{pmatrix} \in \mathbb{R}^{2 \times 8}$$

**Indexing convention** for a 2D array: `b[i, j]` is row $i$, column $j$. NumPy stores the data **C-contiguously** by default: row 0 first, then row 1, with elements within each row adjacent in memory.

In [83]:
b[0]

array([0., 1., 2., 3., 4., 5., 6., 7.])

`b[0]` selects **row 0** (8 elements). When we omit one or more indices, NumPy fills in `:` for them -- so `b[0]` is shorthand for `b[0, :]`.

In [84]:
b[0, 2]

np.float64(2.0)

`b[0, 2]` selects the scalar at row 0, column 2 -- the value **2.0**. This comma-separated indexing is the major ergonomic improvement over nested-list `m[0][2]`.

In [85]:
b[:, 1]

array([1., 2.])

`b[:, 1]` selects **column 1** (all rows, column 1) → `[1., 2.]`. With nested lists, this required a comprehension; with NumPy, a single colon does the job. **Column slicing is the bread-and-butter operation of feature engineering** -- selecting a single feature from a `(n_samples, n_features)` matrix.

In [86]:
b.sum()

np.float64(84.0)

Summing **everything**: $\sum_{i, j} b_{i, j} = (0 + 1 + ... + 7) + (0 + 2 + ... + 14) = 28 + 56 = 84$. The total reduction collapses all axes to a scalar.

In [87]:
b.sum(axis=0)

array([ 0.,  3.,  6.,  9., 12., 15., 18., 21.])

**`axis=0`** means **collapse rows**, leaving columns. The result is a length-8 vector with the column sums:

$$\left( b_{0,j} + b_{1,j} \right)_{j=0}^{7} = (0+0,\, 1+2,\, 2+4,\, ...,\, 7+14) = (0, 3, 6, 9, 12, 15, 18, 21)$$

**Mnemonic**: `axis=0` is the **first** axis (rows in 2D). Reducing along axis 0 *eliminates* axis 0 from the shape. Shape `(2, 8)` → `(8,)`.

In [88]:
b.sum(axis=1)

array([28., 56.])

**`axis=1`** collapses columns, leaving rows:

$$\left( \sum_{j=0}^{7} b_{i, j} \right)_{i=0}^{1} = (0 + 1 + ... + 7,\, 0 + 2 + ... + 14) = (28, 56)$$

Shape `(2, 8)` → `(2,)`. **The cardinal rule of NumPy reductions: the axis you specify is the axis that disappears.** This generalizes to higher dimensions: `arr.sum(axis=(0, 2))` collapses axes 0 and 2 simultaneously, leaving only axis 1.

**ML application:** in deep learning, `sum(axis=-1)` is everywhere -- sum over the feature dimension, leaving the batch dimension untouched. Combined with broadcasting, this is the entire toolkit for batch-aware loss computation.

#### Pre-allocating arrays

When the values are not yet known but the shape is, we use **constructors** that allocate the buffer up front.

In [89]:
c = np.zeros((2, 3), dtype='i', order='C')
c

array([[0, 0, 0],
       [0, 0, 0]], dtype=int32)

**`np.zeros(shape, dtype, order)`** allocates a zero-initialized array. Parameters:

- **`shape=(2, 3)`** -- a 2-row, 3-column matrix.
- **`dtype='i'`** -- 32-bit signed integer (`int32`).
- **`order='C'`** -- row-major memory layout (the default; the alternative `'F'` is column-major / Fortran-style).

A `(2, 3)` `int32` array uses $2 \cdot 3 \cdot 4 = 24$ bytes. Zero-initialization adds a `memset` pass which is essentially free on modern hardware ($\sim 100$ GB/s memory bandwidth).

In [90]:
c = np.ones((2, 3, 4), dtype='i', order='C')
c

array([[[1, 1, 1, 1],
        [1, 1, 1, 1],
        [1, 1, 1, 1]],

       [[1, 1, 1, 1],
        [1, 1, 1, 1],
        [1, 1, 1, 1]]], dtype=int32)

**`np.ones`** -- same API, fills with 1 instead of 0. The shape `(2, 3, 4)` makes a rank-3 tensor: 2 layers, each $3 \times 4$. Total elements: $2 \cdot 3 \cdot 4 = 24$.

**ML pattern:** `np.ones((batch_size, seq_len))` is a typical attention mask; `np.ones((n_classes,)) / n_classes` initializes a uniform prior for Bayesian classifiers.

In [91]:
d = np.zeros_like(c, dtype='f16', order='C')
d

array([[[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]],

       [[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]]], dtype=float128)

**`np.zeros_like(template, dtype, order)`** copies the **shape** of `c` but uses a different dtype -- here `float128` (extended precision, 16 bytes per element on Linux/Mac).

**Caveat:** `float128` (a.k.a. `np.longdouble`) is **platform-dependent**. On Linux/Mac/x86_64 it is typically 80-bit IEEE extended precision padded to 128 bits; on Windows it is identical to `float64`; on ARM Macs it may be true 128-bit quadruple precision. **Never rely on `float128` for cross-platform reproducibility.** For high precision, use Python's `decimal.Decimal` (Chapter 3) or the `mpmath` library.

In [92]:
d = np.ones_like(c, dtype='f16', order='C')
d

array([[[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]],

       [[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]]], dtype=float128)

`np.ones_like` is the symmetric counterpart. The `_like` family (`zeros_like`, `ones_like`, `empty_like`, `full_like`) makes shape-matching trivial -- crucial when chaining operations where the output shape depends on the input.

In [93]:
e = np.empty((2, 3, 2))
e

array([[[4.37948893e-315, 0.00000000e+000],
        [2.10077583e-312, 6.79038654e-313],
        [2.22809558e-312, 2.14321575e-312]],

       [[2.35541533e-312, 6.79038654e-313],
        [2.22809558e-312, 2.14321575e-312],
        [2.46151512e-312, 2.41907520e-312]]])

**`np.empty`** allocates **uninitialized memory**. The values shown above are *whatever bits happened to be in the memory pages we got back*. Each run of this cell will produce different garbage values.

**Why use `empty` over `zeros`?** Performance: skipping the zero-fill saves $\sim 100$ MB/s overhead on a million-element allocation. **Use only when you will overwrite every element immediately afterward.**

**ML pitfall:** if you forget to fill the array, you may train on or report uninitialized memory -- including values like `NaN` (visible above) which propagate through arithmetic. Always prefer `np.zeros` unless you have profiled and confirmed the speed difference matters.

In [94]:
f = np.empty_like(c)
f

array([[[886418428,         0,         0,         0],
        [      117,        99,       104,        32],
        [      102,       105,       108,       101]],

       [[       32,       111,       114,        32],
        [      100,       105,       114,       101],
        [       99,       116,       111,       114]]], dtype=int32)

`empty_like` for an `int32` template: the garbage is now displayed as **arbitrary 32-bit signed integers**, both positive and negative. The values are completely unpredictable and depend on what previous program left in those memory pages.

**Note:** these specific numbers will be different every time you re-run this cell. The notebook displays *one* sample from the distribution of "uninitialized memory states".

In [95]:
np.eye(5)

array([[1., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0.],
       [0., 0., 1., 0., 0.],
       [0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 1.]])

**`np.eye(n)`** creates the $n \times n$ identity matrix:

$$\mathbf{I}_n = \begin{pmatrix} 1 & 0 & \cdots & 0 \\ 0 & 1 & \cdots & 0 \\ \vdots & \vdots & \ddots & \vdots \\ 0 & 0 & \cdots & 1 \end{pmatrix}, \quad (\mathbf{I}_n)_{ij} = \delta_{ij}$$

The **identity matrix** is the multiplicative identity of matrix multiplication: $\mathbf{I} \mathbf{X} = \mathbf{X} \mathbf{I} = \mathbf{X}$ for any compatible $\mathbf{X}$. It appears in ML as:

- The starting point for **Tikhonov regularization**: $(\mathbf{X}^\top \mathbf{X} + \lambda \mathbf{I})^{-1}$ in ridge regression.
- The covariance of an **isotropic Gaussian**: $\mathcal{N}(\boldsymbol{\mu}, \sigma^2 \mathbf{I})$.
- The **identity initialization** for recurrent neural networks (Le et al. 2015).

In [96]:
g = np.linspace(5, 15, 12)
g

array([ 5.        ,  5.90909091,  6.81818182,  7.72727273,  8.63636364,
        9.54545455, 10.45454545, 11.36363636, 12.27272727, 13.18181818,
       14.09090909, 15.        ])

**`np.linspace(start, stop, num)`** creates `num` **evenly spaced points** including both endpoints:

$$g_i = 5 + i \cdot \frac{15 - 5}{12 - 1} = 5 + i \cdot \frac{10}{11}, \quad i = 0, 1, ..., 11$$

The step size is $\frac{10}{11} \approx 0.909$ -- reflected in the spacing between adjacent values. With 12 points, there are **11 intervals**, hence the divisor $11$ (not 12).

**Compare with `np.arange`:**

| | `arange` | `linspace` |
|---|---|---|
| Specifies | step size | number of points |
| Endpoint | exclusive | inclusive (default) |
| Float behavior | rounding errors accumulate | numerically clean |

**Rule:** use `linspace` when you care about exact endpoints (plotting domains, evaluation grids); use `arange` when you care about exact step size (integer indices, time steps).

#### NumPy dtypes

NumPy's dtype system is richer than Python's. Each `ndarray` carries one dtype, applied uniformly to all elements. The major dtype families:

| Code | Description | Example | Bytes |
|---|---|---|---|
| `?` | Boolean | `?` | 1 |
| `i` | Signed integer | `i8` (`int64`) | 8 |
| `u` | Unsigned integer | `u8` (`uint64`) | 8 |
| `f` | Floating point | `f8` (`float64`) | 8 |
| `c` | Complex floating point | `c16` (`complex128`) | 16 |
| `m` | timedelta | `m8[ns]` | 8 |
| `M` | datetime | `M8[ns]` | 8 |
| `O` | Python object (pointer) | `O` | 8 (pointer) |
| `U` | Unicode string | `U24` | 24 × 4 = 96 |
| `V` | Raw bytes (void) | `V12` | 12 |

**ML rules of thumb:**

- **Default to `float64`** for CPU-side preprocessing -- numerical accuracy beats memory savings.
- **Cast to `float32` before GPU upload** -- modern GPUs are 2x faster on `float32` than `float64` (and Tensor Cores need `float16` / `bfloat16`).
- **Use `int32` for indices** when memory matters; `int64` is the default but rarely needed for $< 2 \times 10^9$ samples.
- **Avoid `object` dtype** -- it stores Python pointers, defeating vectorization.

### 4.2.3 Metainformation

Every `ndarray` exposes attributes describing its structure.

In [97]:
g.size

12

**`size`** is the total number of elements: $12$ for our linspace array. Equal to the product of the shape: $\prod_i \text{shape}_i$.

In [98]:
g.itemsize

8

**`itemsize`** is the number of bytes per element: **8 bytes** for `float64`. This matches the C `sizeof(double)` and is fixed by the dtype.

In [99]:
g.ndim

1

**`ndim`** is the **rank** -- the number of dimensions. `g` is a 1D vector, so `ndim = 1`. A `(3, 4)` matrix has `ndim = 2`. Tensor names align with `ndim`:

- 0D = scalar
- 1D = vector
- 2D = matrix
- 3D, 4D, ... = tensor (in the loose ML sense)

In [100]:
g.shape

(12,)

**`shape`** is a `tuple` of dimension lengths. For `g`, it is `(12,)` -- the trailing comma is Python's syntax for a 1-element tuple, distinguishing it from `(12)` which is just the integer 12 in parentheses.

Notice the connection back to Chapter 3: a tuple is **immutable**, which is why `shape` is read-only. To change the shape we must use `reshape` (next subsection), which returns a new view.

In [101]:
g.dtype

dtype('float64')

**`dtype`** is the data type object. For our `linspace`, it's `float64` -- the NumPy default for floating-point arrays. The dtype encapsulates: type code, byte size, byte order, and (for structured arrays) field names.

In [102]:
g.nbytes

96

**`nbytes`** is the total memory footprint in bytes: $\text{size} \times \text{itemsize} = 12 \times 8 = 96$ bytes.

**Memory budgeting rule:**

$$\text{nbytes} = \prod_i \text{shape}_i \times \text{itemsize}$$

For a `(1024, 1024, 3)` `float32` image batch: $1024 \times 1024 \times 3 \times 4 = 12$ MB per image. A batch of 32 such images is $\sim 384$ MB -- already pushing GPU VRAM limits.

### 4.2.4 Reshaping and resizing

**Reshaping** rearranges the same data into a different shape -- size unchanged. **Resizing** changes the size by truncation or wrap-around.

In [103]:
g = np.arange(15)
g

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14])

A test fixture: 15 sequential integers, $g_i = i$ for $i = 0, ..., 14$.

In [104]:
g.shape

(15,)

Or via the functional form:

In [105]:
np.shape(g)

(15,)

Both forms return the shape `(15,)`. The functional form `np.shape(g)` is occasionally useful for objects that may not be ndarrays (e.g., Python lists -- it works on those too, returning `()` for scalars).

In [106]:
g.reshape((3, 5))

array([[ 0,  1,  2,  3,  4],
       [ 5,  6,  7,  8,  9],
       [10, 11, 12, 13, 14]])

**`reshape((3, 5))`** rearranges 15 elements into a $3 \times 5$ matrix, **filling row-by-row** (C-order) by default:

$$\begin{pmatrix} 0 & 1 & 2 & 3 & 4 \\ 5 & 6 & 7 & 8 & 9 \\ 10 & 11 & 12 & 13 & 14 \end{pmatrix}$$

The **only constraint**: $3 \times 5 = 15$ must equal the original size. Otherwise NumPy raises `ValueError`.

**The killer feature:** `reshape` returns a **view** sharing memory with `g`, not a copy. No data movement, no allocation -- just a different shape descriptor. This makes `reshape` essentially free for any size of array.

In [107]:
h = g.reshape((5, 3))
h

array([[ 0,  1,  2],
       [ 3,  4,  5],
       [ 6,  7,  8],
       [ 9, 10, 11],
       [12, 13, 14]])

A different reshape: $5 \times 3$. Same 15 elements, same row-major fill order.

We assign to `h` because we'll use this matrix repeatedly in the next subsections.

In [108]:
h.T

array([[ 0,  3,  6,  9, 12],
       [ 1,  4,  7, 10, 13],
       [ 2,  5,  8, 11, 14]])

**`h.T`** is the **transpose**: rows become columns and vice versa. For our $5 \times 3$ matrix:

$$\mathbf{h}^\top = \begin{pmatrix} 0 & 3 & 6 & 9 & 12 \\ 1 & 4 & 7 & 10 & 13 \\ 2 & 5 & 8 & 11 & 14 \end{pmatrix} \in \mathbb{R}^{3 \times 5}$$

In general $(\mathbf{A}^\top)_{ij} = \mathbf{A}_{ji}$. Like `reshape`, transpose returns a view -- no data is copied. The view simply traverses the original data in column-major order.

**Numerical consequence:** `h.T` is no longer C-contiguous! Operations on it may be slower than on `h` itself if they assume row-major access. This is the bridge to the memory-layout discussion at the end of the chapter.

In [109]:
h.transpose()

array([[ 0,  3,  6,  9, 12],
       [ 1,  4,  7, 10, 13],
       [ 2,  5,  8, 11, 14]])

`h.transpose()` is the **method form** of `.T` -- identical result. The method takes optional axis-permutation arguments for higher-rank tensors:

```python
arr.transpose(2, 0, 1)   # axes 0,1,2 → 2,0,1 in result
```

For convolutional networks with `(N, H, W, C)` (TensorFlow/Keras) vs `(N, C, H, W)` (PyTorch) layouts, `.transpose(0, 3, 1, 2)` is the canonical conversion.

#### Resizing -- destructive shape change

`np.resize` is fundamentally different from `reshape`: it **changes the total number of elements**, truncating or repeating to fit the new shape.

In [110]:
g

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14])

Our 15-element fixture, repeated for clarity.

In [111]:
np.resize(g, (3, 1))

array([[0],
       [1],
       [2]])

**Down-sizing**: requesting only $3 \times 1 = 3$ elements means NumPy keeps **the first 3 elements** of `g` and discards the rest. Information is **destroyed**. A subtle but real risk in production code -- if you intend `reshape` and accidentally type `resize`, you silently lose data.

In [112]:
np.resize(g, (1, 5))

array([[0, 1, 2, 3, 4]])

Down-sized to $1 \times 5 = 5$ elements: the first 5 of `g`. Note the **2D shape** -- a single-row matrix, not a 1D vector. Shape `(1, 5)` and shape `(5,)` are different.

In [113]:
np.resize(g, (2, 5))

array([[0, 1, 2, 3, 4],
       [5, 6, 7, 8, 9]])

$2 \times 5 = 10$ elements: the first 10 of `g`, reshaped to $2 \times 5$.

In [114]:
n = np.resize(g, (5, 4))
n

array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11],
       [12, 13, 14,  0],
       [ 1,  2,  3,  4]])

**Up-sizing**: requesting $5 \times 4 = 20$ elements when only 15 are available -- NumPy **wraps around** and reuses the data from the start. The last row contains $0, 1, 2, 3, 4$ (re-cycled).

**This is rarely what you want.** The wrap-around behavior is so unusual that `np.resize` is virtually never used in modern code. Use `reshape` for shape changes, `np.tile` for explicit replication, and `np.pad` for padded enlargement.

#### Stacking

Combining arrays along an axis is a fundamental operation. NumPy provides `hstack` (horizontal), `vstack` (vertical), and the more general `concatenate`.

In [115]:
h

array([[ 0,  1,  2],
       [ 3,  4,  5],
       [ 6,  7,  8],
       [ 9, 10, 11],
       [12, 13, 14]])

Recall `h` is the $5 \times 3$ matrix.

In [116]:
np.hstack((h, 2 * h))

array([[ 0,  1,  2,  0,  2,  4],
       [ 3,  4,  5,  6,  8, 10],
       [ 6,  7,  8, 12, 14, 16],
       [ 9, 10, 11, 18, 20, 22],
       [12, 13, 14, 24, 26, 28]])

**`hstack`** stacks **horizontally** (along axis 1): the result has the same number of rows but doubled column count. Shape: $(5, 3) + (5, 3) \to (5, 6)$.

The "connecting" dimension is axis 0 (rows): both matrices must have the same number of rows for `hstack` to work.

**ML use:** `hstack` is the canonical way to **add features** to a feature matrix:

```python
X_full = np.hstack([X_numerical, X_one_hot_encoded])
```

In [117]:
np.vstack((h, 0.5 * h))

array([[ 0. ,  1. ,  2. ],
       [ 3. ,  4. ,  5. ],
       [ 6. ,  7. ,  8. ],
       [ 9. , 10. , 11. ],
       [12. , 13. , 14. ],
       [ 0. ,  0.5,  1. ],
       [ 1.5,  2. ,  2.5],
       [ 3. ,  3.5,  4. ],
       [ 4.5,  5. ,  5.5],
       [ 6. ,  6.5,  7. ]])

**`vstack`** stacks **vertically** (along axis 0): same number of columns, doubled rows. Shape: $(5, 3) + (5, 3) \to (10, 3)$.

Notice the dtype was promoted to `float64` because `0.5 * h` produces floats. Type promotion rules apply: `int + float → float`, `float32 + float64 → float64`, etc.

**ML use:** `vstack` adds **samples** to a feature matrix -- the canonical operation for combining train/test/val splits or merging data from multiple sources.

#### Flattening

The reverse of `reshape((m, n))` from a 1D vector: collapse a multi-dimensional array back to 1D.

In [118]:
h

array([[ 0,  1,  2],
       [ 3,  4,  5],
       [ 6,  7,  8],
       [ 9, 10, 11],
       [12, 13, 14]])

Reminder of the $5 \times 3$ matrix.

In [119]:
h.flatten()

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14])

**`flatten()`** returns a 1D **copy** of the array, traversing in C order (row by row): $0, 1, 2,\, 3, 4, 5,\, 6, 7, 8,\, 9, 10, 11,\, 12, 13, 14$.

A copy is allocated -- modifying the result does not affect `h`. For a non-allocating alternative, see `ravel()` below.

In [120]:
h.flatten(order='C')

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14])

**`order='C'`** is the default (row-major). The result matches `flatten()` with no argument.

In [121]:
h.flatten(order='F')

array([ 0,  3,  6,  9, 12,  1,  4,  7, 10, 13,  2,  5,  8, 11, 14])

**`order='F'`** flattens in column-major (Fortran) order -- column 0 first, then column 1, then column 2:

$$0, 3, 6, 9, 12 \;|\; 1, 4, 7, 10, 13 \;|\; 2, 5, 8, 11, 14$$

This is exactly the layout of `h.T.flatten('C')` -- transposing then C-flattening is equivalent to F-flattening the original.

**When does C/F order matter for ML?** When importing data from Fortran-based libraries (LAPACK, BLAS), Julia, R, or MATLAB -- all of which use column-major by default. NumPy ↔ MATLAB data exchange should always specify `order='F'` for the MATLAB side.

In [122]:
for i in h.flat:
    print(i, end=',')

0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,

**`h.flat`** is a 1D iterator over the array (C order), without materializing a flat copy. Memory-efficient for very large arrays where flattening would double the footprint.

The output is the **15 elements** comma-separated, with the row order following C convention.

In [123]:
for i in h.ravel(order='C'):
    print(i, end=',')

0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,

And in F-order, the contents are the same as `flatten(order='F')`:

In [124]:
for i in h.ravel(order='F'):
    print(i, end=',')

0,3,6,9,12,1,4,7,10,13,2,5,8,11,14,

**`ravel()`** is similar to `flatten()` but **returns a view when possible** (and a copy only when necessary -- e.g., if the requested order requires non-contiguous traversal).

| Method | Returns | Cost |
|---|---|---|
| `arr.flatten()` | always a copy | $O(n)$ allocation + memcpy |
| `arr.ravel()` | view if contiguous, else copy | $O(1)$ in best case |
| `arr.flat` | iterator (no allocation) | $O(1)$ |

**Rule:** prefer `ravel()` over `flatten()` unless you specifically need an independent copy. The savings on a 1 GB array are significant: a $\sim 3$ ms allocation vs a $\sim 0$ ms view.

### 4.2.5 Boolean arrays

Element-wise comparisons return Boolean arrays of the same shape. These are the key to **vectorized filtering** -- one of the most powerful idioms in NumPy.

In [125]:
h

array([[ 0,  1,  2],
       [ 3,  4,  5],
       [ 6,  7,  8],
       [ 9, 10, 11],
       [12, 13, 14]])

Reminder.

In [126]:
h > 8

array([[False, False, False],
       [False, False, False],
       [False, False, False],
       [ True,  True,  True],
       [ True,  True,  True]])

**Element-wise greater-than**: each element of `h` is compared to the scalar `8`, producing a Boolean array of the **same shape** $(5, 3)$. The result has 6 `True` values (entries 9, 10, 11, 12, 13, 14) and 9 `False` values.

This is the algorithmic heart of decision trees: at each node, a comparison `feature[j] > threshold` produces a Boolean mask used to partition the samples. NumPy can evaluate such comparisons on millions of samples in milliseconds.

In [127]:
h <= 7

array([[ True,  True,  True],
       [ True,  True,  True],
       [ True,  True, False],
       [False, False, False],
       [False, False, False]])

**Less-than-or-equal**: 8 entries are $\leq 7$ (the values 0 through 7), producing 8 `True`s. Combined with `>`, these comparisons partition all elements into two complementary halves.

In [128]:
h == 5

array([[False, False, False],
       [False, False,  True],
       [False, False, False],
       [False, False, False],
       [False, False, False]])

**Equality**: only the single entry containing `5` is `True`. The Boolean array is **mostly false** -- a sparse pattern.

**Float warning:** `h == 5.0` would also work here because both sides are integers. But `np.array([0.1 + 0.2]) == np.array([0.3])` returns `False` due to floating-point rounding! Always use `np.isclose(a, b)` for floats.

In [129]:
(h == 5).astype(int)

array([[0, 0, 0],
       [0, 0, 1],
       [0, 0, 0],
       [0, 0, 0],
       [0, 0, 0]])

**`astype(int)`** casts the Boolean array to integers: `True → 1`, `False → 0`. The result is a "0/1 indicator matrix" -- exactly the form of one-hot encoded labels in classification.

**ML connection:** computing accuracy is one line:

```python
accuracy = (y_pred == y_true).astype(int).mean()
```

The `astype` is even unnecessary -- `.mean()` of a Boolean array implicitly casts. This single idiom replaces several lines of pure-Python looping.

In [130]:
(h > 4) & (h <= 12)

array([[False, False, False],
       [False, False,  True],
       [ True,  True,  True],
       [ True,  True,  True],
       [ True, False, False]])

**Combined Boolean**: `&` is element-wise AND. The mask is `True` where $4 < h_{ij} \leq 12$. Counting: entries 5, 6, 7, 8, 9, 10, 11, 12 satisfy the condition → 8 `True`s.

**Critical syntax warning:** in NumPy, you must use **bitwise operators** `&` (AND), `|` (OR), `~` (NOT) -- *not* the Python keywords `and`, `or`, `not`. The latter try to evaluate the truthiness of the entire array and raise `ValueError`. Additionally, due to Python's operator precedence rules, **always parenthesize each comparison**: `(a > 4) & (a <= 12)`, not `a > 4 & a <= 12` (which Python parses as `a > (4 & a) <= 12`).

#### Boolean indexing

A Boolean array can index another array of the same shape, selecting only the elements where the mask is `True`. The result is **flattened** to 1D.

In [131]:
h[h > 8]

array([ 9, 10, 11, 12, 13, 14])

**6 elements** greater than 8: $\{9, 10, 11, 12, 13, 14\}$. The result is 1D regardless of the input shape -- there's no canonical 2D shape that preserves "all elements > 8" while keeping rectangular structure.

This is the canonical syntax for **filtering rows / elements**:

```python
positives = X[y == 1]      # all positive-class samples
recent = df[df['date'] > '2024-01-01']   # pandas, same idea
```

In [132]:
h[(h > 4) & (h <= 12)]

array([ 5,  6,  7,  8,  9, 10, 11, 12])

**Range filter**: 8 elements in $(4, 12]$. The compound mask is built once, then applied. NumPy's filtering performance is far superior to pure-Python list comprehensions for this kind of work -- a $10^7$-element array can be filtered in milliseconds.

In [133]:
h[(h < 4) | (h >= 12)]

array([ 0,  1,  2,  3, 12, 13, 14])

**OR mask**: elements outside the inner range, $7$ elements total. The complement of an interval condition often appears in **outlier removal**:

```python
clean = X[~((X > q1) & (X < q3))]   # keep only outliers (anti-IQR filter)
```

#### `np.where`: vectorized if/else

`np.where(condition, value_if_true, value_if_false)` is the vectorized ternary operator.

In [134]:
np.where(h > 7, 1, 0)

array([[0, 0, 0],
       [0, 0, 0],
       [0, 0, 1],
       [1, 1, 1],
       [1, 1, 1]])

For each entry of `h`, return `1` if `h > 7` else `0`. The output preserves the **2D shape** of `h` (unlike Boolean indexing which flattens), with 7 ones and 8 zeros.

`np.where` with constant arms is essentially `(condition).astype(int)` * value. Its real power comes from non-constant arms (next examples).

In [135]:
np.where(h % 2 == 0, 'even', 'odd')

array([['even', 'odd', 'even'],
       ['odd', 'even', 'odd'],
       ['even', 'odd', 'even'],
       ['odd', 'even', 'odd'],
       ['even', 'odd', 'even']], dtype='<U4')

For each entry, return `'even'` or `'odd'` based on parity. The dtype `'<U4'` is automatic -- NumPy chose 4 characters, the length of the longer string `'even'`.

**ML application:** quick categorical labelling without an explicit loop. For binary thresholding of probabilities to class labels:

```python
y_pred_label = np.where(y_pred_proba >= 0.5, 'positive', 'negative')
```

In [136]:
np.where(h <= 7, h * 2, h / 2)

array([[ 0. ,  2. ,  4. ],
       [ 6. ,  8. , 10. ],
       [12. , 14. ,  4. ],
       [ 4.5,  5. ,  5.5],
       [ 6. ,  6.5,  7. ]])

**The full power**: each arm is itself an array operation. For each $h_{ij}$:

$$\text{result}_{ij} = \begin{cases} 2 h_{ij} & \text{if } h_{ij} \leq 7 \\ h_{ij} / 2 & \text{otherwise} \end{cases}$$

The output dtype is `float64` because the `else` branch produces floats.

**ML pattern:** "huberize" or "soften" arrays -- e.g., the **Huber loss** combines two regimes (quadratic near zero, linear far away), implementable in one `np.where`:

$$L_\delta(e) = \begin{cases} \frac{1}{2} e^2 & |e| \leq \delta \\ \delta (|e| - \frac{1}{2} \delta) & |e| > \delta \end{cases}$$

```python
def huber(e, delta=1.0):
    abs_e = np.abs(e)
    return np.where(abs_e <= delta, 0.5 * e**2, delta * (abs_e - 0.5 * delta))
```

### 4.2.6 Speed comparison: Python vs NumPy

The motivation for NumPy in one experiment.

In [137]:
I = 5000

%time mat = [[np.random.standard_normal() for j in range(I)] for i in range(I)]

CPU times: user 16.1 s, sys: 468 ms, total: 16.6 s
Wall time: 16.8 s


Building a $5000 \times 5000$ matrix of $25{,}000{,}000$ standard-normal samples via a **pure-Python nested list comprehension** takes about **22 seconds** on this machine. (The book reports 1.34s on a fast workstation -- exact wall time depends on CPU.)

Each call to `np.random.standard_normal()` in this loop creates a 0-dimensional NumPy array, then a Python float wrapper -- both massively wasteful given that the function itself is C-vectorized when called with a shape argument.

In [138]:
mat[0][:5]

[-1.7497654730546974,
 0.34268040332750216,
 1.153035802563644,
 -0.25243603652138985,
 0.9813207869512316]

The first 5 samples from row 0. These are pseudo-random draws from $\mathcal{N}(0, 1)$. With **25 million samples**, the empirical mean should be $\sim 0$ (to within $1 / \sqrt{25 \times 10^6} \approx 2 \times 10^{-4}$ standard error).

In [139]:
%time sum([sum(l) for l in mat])

CPU times: user 268 ms, sys: 40 µs, total: 269 ms
Wall time: 269 ms


-4715.2052785152

Summing all $25 \times 10^6$ elements via nested Python `sum()` takes **~270 ms**. The total $\approx 27.98$ -- well within the expected $\pm 5\sqrt{n} \approx \pm 25{,}000$ standard deviation, so the per-sample mean is $\sim 1.1 \times 10^{-6}$ -- consistent with $\bar{X} \approx 0$.

**Why so slow?** The outer `sum()` performs $5000$ nested-list traversals. Each inner `sum()` does $5000$ Python-level additions, each requiring object allocation and reference counting. Total: $\sim 5 \times 10^7$ Python-level operations.

In [140]:
mem_lists = sum([sys.getsizeof(l) for l in mat])
mem_lists

209400000

Each inner list (5000 floats) consumes about $\frac{209.4 \text{ MB}}{5000} \approx 41{,}880$ bytes for the list overhead alone -- not counting the float objects themselves. The full footprint including each `float` object's $\sim 28$ bytes is approximately:

$$\underbrace{2.1 \times 10^8}_{\text{list pointers}} + \underbrace{2.5 \times 10^7 \times 28}_{\text{float objects}} = 0.21 \text{ GB} + 0.7 \text{ GB} \approx 0.9 \text{ GB}$$

Per element: $\sim 36$ bytes, vs the **8 bytes** a `float64` array would use. **A 4.5x memory amplification.**

In [141]:
%time mat = np.random.standard_normal((I, I))

CPU times: user 1.16 s, sys: 152 ms, total: 1.31 s
Wall time: 1.32 s


**The NumPy version**: a single function call generating the same $5000 \times 5000 = 25 \times 10^6$ samples takes **~1.76 seconds** -- a **~12.5x speedup** over the Python loop. (The book reports 1.21s -- ratio is similar.)

The speed comes from:

1. **No Python-level loop**: NumPy's PRNG generates millions of samples in a single C call.
2. **No per-element object overhead**: contiguous `float64` buffer.
3. **SIMD vectorization**: modern PRNGs can fill 4-8 doubles per CPU cycle.

In [142]:
%time mat.sum()

CPU times: user 22.4 ms, sys: 0 ns, total: 22.4 ms
Wall time: 22.8 ms


np.float64(2495.8562726344735)

Summing the same $25 \times 10^6$ elements with `mat.sum()` takes **~33 ms** -- a **~8x speedup** over the Python `sum()`. The single C-level loop processes ~$7.5 \times 10^8$ floats per second, near the memory bandwidth limit.

Note the empirical sum is **2812.5** here (vs 27.98 from the Python version) because we're summing a **different** random matrix -- the seed wasn't reset.

In [143]:
mat.nbytes

200000000

**Exactly $200{,}000{,}000$ bytes** = $25 \times 10^6 \times 8$ bytes/double = 200 MB. Predictable and minimal -- precisely the dense storage budget with no overhead.

In [144]:
sys.getsizeof(mat)

200000128

Python's `sys.getsizeof` adds a tiny **128-byte header** (the NumPy object metadata: dtype, shape, strides, base pointer). For a 200 MB array, that's a $6 \times 10^{-5}$ % overhead -- negligible.

**Net comparison:**

| Metric | Pure Python | NumPy | Speedup / saving |
|---|---|---|---|
| Build time | ~22 s | ~1.76 s | **~12.5x faster** |
| Sum time | ~270 ms | ~33 ms | **~8x faster** |
| Memory | ~900 MB | ~200 MB | **~4.5x smaller** |

For larger arrays the gaps widen further -- the memory-bound regime favors NumPy increasingly as data scale grows.

> **Using NumPy Arrays.** The use of NumPy for array-based operations and algorithms generally results in compact, easily readable code and significant performance improvements over pure Python code.

## 4.3 Structured NumPy Arrays

Regular `ndarray`s require a **single dtype** for all elements. **Structured arrays** relax this: each "column" can have its own dtype, like a table in a SQL database. They are the predecessor of (and inspiration for) pandas DataFrames.

In [145]:
dt = np.dtype([('Name', 'S10'),
               ('Age', 'i4'),
               ('Height', 'f'),
               ('Children/Pets', 'i4', 2)])

We define a composite dtype with **4 fields**:

| Field | Type code | Description |
|---|---|---|
| `Name` | `S10` | 10-byte ASCII string |
| `Age` | `i4` | 32-bit signed integer |
| `Height` | `f` | 32-bit float |
| `Children/Pets` | `i4`, 2 | A pair of int32 (a sub-array per row) |

This is the "schema" -- an analog of a `CREATE TABLE` statement in SQL. Total bytes per record: $10 + 4 + 4 + (4 \times 2) = 26$ bytes (NumPy may pad to align).

In [146]:
dt

dtype([('Name', 'S10'), ('Age', '<i4'), ('Height', '<f4'), ('Children/Pets', '<i4', (2,))])

Inspecting `dt`: each field name maps to a (sub-dtype, optional shape) pair. The `<` prefix denotes **little-endian byte order** (the native order on x86/ARM). The `i4`/`f4` are the explicit byte sizes.

**Production warning:** structured arrays were the original "tabular data" solution in NumPy, but they are essentially deprecated for analytical work. **Use pandas DataFrames** for tabular data -- richer API, faster groupby, better string handling, full indexing semantics. Structured arrays remain useful only for fixed binary file formats (e.g., parsing trading-platform tick records).

In [147]:
dt = np.dtype({'names': ['Name', 'Age', 'Height', 'Children/Pets'],
                'formats': 'O int float int,int'.split()})

In [148]:
dt

dtype([('Name', 'O'), ('Age', '<i8'), ('Height', '<f8'), ('Children/Pets', [('f0', '<i8'), ('f1', '<i8')])])

An alternative dictionary-based dtype constructor with different types:

- `Name` → `O` (Python object pointer) -- variable-length strings, no truncation.
- `Age` → `int` (default `int64`).
- `Height` → `float` (default `float64`).
- `Children/Pets` → `int,int` (a 2-field nested struct with default `int64` fields).

Notice the ergonomic shift: `O` (object) handles any-length strings but **defeats vectorization** -- arithmetic on object arrays runs at Python speed.

In [149]:
s = np.array([('Smith', 45, 1.83, (0, 1)),
              ('Jones', 53, 1.72, (2, 2))], dtype=dt)
s

array([('Smith', 45, 1.83, (0, 1)), ('Jones', 53, 1.72, (2, 2))],
      dtype=[('Name', 'O'), ('Age', '<i8'), ('Height', '<f8'), ('Children/Pets', [('f0', '<i8'), ('f1', '<i8')])])

A 2-record table. Each row is a tuple matching the schema: `(Name, Age, Height, Children/Pets)`. The display reflects the structured layout.

This is conceptually identical to:

```sql
CREATE TABLE people (
    Name           VARCHAR,
    Age            BIGINT,
    Height         DOUBLE,
    Children_Pets  STRUCT<f0 BIGINT, f1 BIGINT>
);
INSERT INTO people VALUES
    ('Smith', 45, 1.83, ROW(0, 1)),
    ('Jones', 53, 1.72, ROW(2, 2));
```

In [150]:
type(s)

numpy.ndarray

Despite the table-like behavior, `s` is still an `ndarray` -- a 1D array whose elements are records. All `ndarray` methods apply.

In [151]:
s['Name']

array(['Smith', 'Jones'], dtype=object)

**Column access by name**: `s['Name']` returns the `Name` column as a 1D array. This is the bridge between structured arrays and pandas Series:

```python
s['Name']                    # NumPy structured
df['Name']                   # pandas DataFrame (same syntax!)
```

pandas inherited this exact accessor convention from structured arrays.

In [152]:
s['Height'].mean()

np.float64(1.775)

Calling `.mean()` on the `Height` column gives the average: $\frac{1.83 + 1.72}{2} = 1.775$.

**Method chaining**: the column extraction returns a regular `ndarray`, so the full NumPy reduction API (`.sum()`, `.std()`, `.argmax()`, etc.) is available -- no different from a pandas Series.

In [153]:
s[0]

np.void(('Smith', 45, 1.83, (0, 1)), dtype=[('Name', 'O'), ('Age', '<i8'), ('Height', '<f8'), ('Children/Pets', [('f0', '<i8'), ('f1', '<i8')])])

**Row access by index**: `s[0]` returns the entire first record. The display shows each field's NumPy scalar type. The record can be unpacked: `name, age, height, kids = s[0]`.

In [154]:
s[1]['Age']

np.int64(53)

**Drilling in**: row 1, then field `'Age'`, gives the scalar `53`. The two-step access pattern `s[i]['field']` exists, but the column-first form `s['Age'][1]` is more idiomatic and slightly faster (avoids materializing an intermediate record).

> **Structured Arrays.** NumPy provides, in addition to regular arrays, structured (and record) arrays that allow the description and handling of table-like data structures with a variety of different data types per (named) column. They bring SQL table-like data structures to Python, with most of the benefits of regular `ndarray` objects (syntax, methods, performance).

## 4.4 Vectorization of Code

**Vectorization** = expressing operations as array-level transformations rather than element-level loops. NumPy was designed around this principle.

### 4.4.1 Basic vectorization

In [155]:
np.random.seed(100)
r = np.arange(12).reshape((4, 3))
s = np.arange(12).reshape((4, 3)) * 0.5

Two test fixtures. We seed the RNG even though `arange` is deterministic -- a habit worth keeping in case future code adds random elements.

In [156]:
r

array([[ 0,  1,  2],
       [ 3,  4,  5],
       [ 6,  7,  8],
       [ 9, 10, 11]])

$r$ is the integer $4 \times 3$ matrix $r_{ij} = 3i + j$.

In [157]:
s

array([[0. , 0.5, 1. ],
       [1.5, 2. , 2.5],
       [3. , 3.5, 4. ],
       [4.5, 5. , 5.5]])

$s$ is the half-step version: $s_{ij} = 0.5 \cdot (3i + j)$. Note the dtype is `float64` because the multiplication by `0.5` triggered type promotion.

In [158]:
r + s

array([[ 0. ,  1.5,  3. ],
       [ 4.5,  6. ,  7.5],
       [ 9. , 10.5, 12. ],
       [13.5, 15. , 16.5]])

**Element-wise addition**: $(r + s)_{ij} = r_{ij} + s_{ij} = 1.5 \cdot r_{ij}$. The shape stays $(4, 3)$. **No loop required** -- the C-level implementation handles all $4 \times 3 = 12$ additions in one call.

**This is the heart of vectorization.** In ML code, we apply this pattern to:

- Computing residuals: `residuals = y - y_hat`.
- Updating weights: `W -= learning_rate * grad_W`.
- Computing the cosine similarity matrix: `(A @ B.T) / (norm_A[:, None] * norm_B[None, :])`.

#### Broadcasting -- the magic that makes vectorization general

Broadcasting lets NumPy combine arrays of **different shapes** without explicit replication. The rules:

1. **Align shapes from the right.** If `a.shape = (4, 3)` and `b.shape = (3,)`, align as `(4, 3)` and `(_, 3)`.
2. **Each dimension must either match or be 1.** `(4, 3)` vs `(3,)` works because the missing dim is treated as 1.
3. **Stretch dimensions of size 1** (logically) to match -- without copying memory.

This produces the rules:

$$\text{broadcast}((m, n), (n,)) = (m, n)$$
$$\text{broadcast}((m, n), (m, 1)) = (m, n)$$
$$\text{broadcast}((m, 1), (1, n)) = (m, n)$$

In [159]:
r + 3

array([[ 3,  4,  5],
       [ 6,  7,  8],
       [ 9, 10, 11],
       [12, 13, 14]])

**Scalar + array**: the scalar `3` is broadcast (logically replicated) to a $(4, 3)$ matrix of 3s, then added element-wise. Each entry of `r` increases by 3.

The "logical replication" is **not physically performed** -- the C loop simply uses the scalar 3 for every iteration. Memory and time cost are the same as `r += 0`.

In [160]:
2 * r

array([[ 0,  2,  4],
       [ 6,  8, 10],
       [12, 14, 16],
       [18, 20, 22]])

**Scalar multiplication**: each element of `r` doubled. Equivalent to `r * 2` (NumPy is commutative for `*` like normal arithmetic).

In [161]:
2 * r + 3

array([[ 3,  5,  7],
       [ 9, 11, 13],
       [15, 17, 19],
       [21, 23, 25]])

**Composed**: $2 r_{ij} + 3$. Two broadcast operations chained -- Python evaluates `2 * r` first (giving an intermediate $(4, 3)$ array), then adds `3` (broadcast). Two passes over the data.

**Optimization tip:** for performance-critical code, replace this with `np.add(np.multiply(r, 2, out=tmp), 3, out=tmp)` to write into a pre-allocated buffer. Or use **Numba / Cython / NumPy ufunc kernels** for fused operations. For most ML preprocessing, the naive form is fast enough.

In [162]:
r

array([[ 0,  1,  2],
       [ 3,  4,  5],
       [ 6,  7,  8],
       [ 9, 10, 11]])

And its shape:

In [163]:
r.shape

(4, 3)

Now build a length-3 vector:

In [164]:
s = np.arange(0, 12, 4)
s

array([0, 4, 8])

A **length-3 vector** $s = (0, 4, 8)$. Its shape `(3,)` aligns with the second axis of `r` (also length 3). Broadcasting is about to happen.

In [165]:
r + s

array([[ 0,  5, 10],
       [ 3,  8, 13],
       [ 6, 11, 16],
       [ 9, 14, 19]])

**Matrix + vector** via broadcasting:

$$(r + s)_{ij} = r_{ij} + s_j$$

The vector `s = (0, 4, 8)` is replicated to **each row** of `r`:

$$\begin{pmatrix} 0 & 1 & 2 \\ 3 & 4 & 5 \\ 6 & 7 & 8 \\ 9 & 10 & 11 \end{pmatrix} + \begin{pmatrix} 0 & 4 & 8 \\ 0 & 4 & 8 \\ 0 & 4 & 8 \\ 0 & 4 & 8 \end{pmatrix} = \begin{pmatrix} 0 & 5 & 10 \\ 3 & 8 & 13 \\ 6 & 11 & 16 \\ 9 & 14 & 19 \end{pmatrix}$$

**ML application**: this is exactly how **bias addition** works in a dense layer:

$$\mathbf{Y} = \mathbf{X} \mathbf{W} + \mathbf{b}, \quad \mathbf{X} \in \mathbb{R}^{N \times d_{\text{in}}}, \; \mathbf{W} \in \mathbb{R}^{d_{\text{in}} \times d_{\text{out}}}, \; \mathbf{b} \in \mathbb{R}^{d_{\text{out}}}$$

The bias `b` (shape `(d_out,)`) broadcasts across all $N$ rows of `XW` (shape `(N, d_out)`), adding the same offset to every sample.

In [166]:
s = np.arange(0, 12, 3)
s

array([0, 3, 6, 9])

A **length-4 vector**. This time the length **doesn't match** either axis of `r` (which has shape `(4, 3)`).

In [167]:
r + s

ValueError: operands could not be broadcast together with shapes (4,3) (4,) 

**`ValueError`**: the shapes `(4, 3)` and `(4,)` don't broadcast. Aligning right:

```
(4, 3)
   (4,)    ← treated as (_, 4)
```

The trailing dimension of `r` is 3, but `s` is length 4. **Broadcasting fails** because neither dimension is 1 nor are they equal.

This is one of the most common NumPy errors. The fix: either transpose, reshape, or align via newaxis.

In [168]:
r.transpose() + s

array([[ 0,  6, 12, 18],
       [ 1,  7, 13, 19],
       [ 2,  8, 14, 20]])

**Fix 1**: transpose `r` to shape `(3, 4)`. Now the trailing axis is 4, matching `s`. Broadcasting succeeds:

$$(r^\top + s)_{ij} = r_{ji} + s_j$$

The vector `s = (0, 3, 6, 9)` is replicated across the **3 rows** of `r.T`. The numbers $0$ through $20$ appear in a 3×4 layout.

In [169]:
sr = s.reshape(-1, 1)
sr

array([[0],
       [3],
       [6],
       [9]])

**Fix 2**: reshape `s` to a column vector (shape `(4, 1)`). The `-1` is a wildcard meaning "infer this dimension from the size". `s.reshape(-1, 1)` always converts a 1D array to a column.

**Common idiom**: `arr.reshape(-1, 1)` to convert features from 1D to a column matrix that scikit-learn estimators expect.

In [170]:
sr.shape

(4, 1)

Confirmed: $(4, 1)$. The dimension of size 1 is the broadcastable one.

In [171]:
r + s.reshape(-1, 1)

array([[ 0,  1,  2],
       [ 6,  7,  8],
       [12, 13, 14],
       [18, 19, 20]])

**Now broadcasting works the other way**: `s` (shape `(4, 1)`) replicates **across columns**, adding a different scalar to each row of `r`:

$$(r + s)_{ij} = r_{ij} + s_i$$

Note the *result is different* from the transpose-fix approach! Mathematically:

- `r.T + s` = column-wise addition (each *column* of `r` gets `s` added).
- `r + s.reshape(-1, 1)` = row-wise addition (each *row* of `r` gets a scalar added).

**ML application**: row-wise broadcasting handles **per-sample normalization**:

```python
mean_per_sample = X.mean(axis=1, keepdims=True)   # shape (N, 1)
X_centered = X - mean_per_sample                  # broadcasts to (N, D)
```

The `keepdims=True` is the key: it preserves the shape `(N, 1)` instead of collapsing to `(N,)`, making broadcasting unambiguous.

#### Custom functions vectorize for free

Pure-arithmetic Python functions automatically work on `ndarray` inputs.

In [172]:
def f(x):
    return 3 * x + 5

f(0.5)

6.5

A scalar input gives a scalar output: $f(0.5) = 3 \cdot 0.5 + 5 = 6.5$.

In [173]:
f(r)

array([[ 5,  8, 11],
       [14, 17, 20],
       [23, 26, 29],
       [32, 35, 38]])

**The same function applied to an `ndarray`**: NumPy substitutes `x = r` and the operations `3 * x + 5` are now element-wise. We get $f$ applied entry-wise to `r`:

$$f(r)_{ij} = 3 r_{ij} + 5$$

**No special "vectorize" decoration is needed**, *as long as*:

1. The function uses only **arithmetic operators** (`+`, `-`, `*`, `/`, `**`, `%`).
2. NumPy ufuncs (`np.exp`, `np.sqrt`, etc.) replace any `math.<func>` calls.
3. The function avoids Python-level branches like `if x > 0: ...` -- those don't broadcast. Use `np.where` instead.

**Production note:** "vectorizes for free" delegates the loop from Python to C. The function `f` is *not* truly vectorized at the CPU level -- NumPy still loops, just in C. For real CPU-level vectorization (SIMD), use **Numba (`@njit`)** or **Cython**, which compile the Python source to machine code.

### 4.4.2 Memory layout

For small arrays, memory layout barely matters. For arrays at scale, it can change runtime by 5-10x.

In [174]:
x = np.random.standard_normal((1000000, 5))
y = 2 * x + 3
C = np.array((x, y), order='C')
F = np.array((x, y), order='F')
x = 0.0; y = 0.0  # free intermediate buffers

We build two rank-3 arrays of shape `(2, 1_000_000, 5)` -- $10^7$ elements each, $80$ MB at `float64`. Same data, different memory layouts:

- **`order='C'`** (row-major / "C-order" / "C-contiguous"): the **last axis varies fastest** in memory. Iterating `arr[i, j, k]` for `k = 0, 1, 2, ...` walks contiguous memory.
- **`order='F'`** (column-major / "F-order" / "Fortran-contiguous"): the **first axis varies fastest**. Iterating `arr[k, j, i]` walks contiguous memory.

**Mental model:** every multi-dimensional array is stored as a 1D buffer plus a "strides" tuple telling NumPy how to translate `(i, j, k)` into a buffer offset.

In [175]:
C[:2].round(2)

array([[[-1.75,  0.34,  1.15, -0.25,  0.98],
        [ 0.51,  0.22, -1.07, -0.19,  0.26],
        [-0.46,  0.44, -0.58,  0.82,  0.67],
        ...,
        [-0.05,  0.14,  0.17,  0.33,  1.39],
        [ 1.02,  0.3 , -1.23, -0.68, -0.87],
        [ 0.83, -0.73,  1.03,  0.34, -0.46]],

       [[-0.5 ,  3.69,  5.31,  2.5 ,  4.96],
        [ 4.03,  3.44,  0.86,  2.62,  3.51],
        [ 2.08,  3.87,  1.83,  4.63,  4.35],
        ...,
        [ 2.9 ,  3.28,  3.33,  3.67,  5.78],
        [ 5.04,  3.6 ,  0.54,  1.65,  1.26],
        [ 4.67,  1.54,  5.06,  3.69,  2.07]]])

A glimpse at `C`'s contents (rounded to 2 decimals). The shape `(2, 1_000_000, 5)` and the layered structure is visible:

- Layer 0: the standard normal samples `x`.
- Layer 1: the linear transform $y = 2x + 3$ -- visibly shifted right by ~3 and scaled up by 2.

For example, `C[0, 0, 0] ≈ 1.76` and `C[1, 0, 0] ≈ 6.53`. Verify: $2 \cdot 1.76 + 3 = 6.52$ ✓ (small rounding diff in the 2nd decimal).

In [176]:
%timeit C.sum()

7.22 ms ± 164 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


And for F-order:

In [177]:
%timeit F.sum()

8.03 ms ± 931 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


**Total sum: layout doesn't matter.** Both take about **6.6 ms**. Reason: a global reduction reads every byte exactly once. The order in which bytes are visited is irrelevant once they're all in cache eventually -- the critical metric is **total memory traffic**, which is the same for both.

(The book reports ~4.3 ms each on a faster machine; ratios remain unity.)

In [178]:
%timeit C.sum(axis=0)

19.5 ms ± 372 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


Or summing over the second-largest axis:

In [179]:
%timeit C.sum(axis=1)

44.3 ms ± 2.98 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


**For C-order: `axis=0` (~26 ms) is faster than `axis=1` (~33 ms).** Why?

Recall the shape is `(2, 1_000_000, 5)`. C-order means the **last axis (5) varies fastest in memory**. The buffer layout is:

```
[ x[0,0,0], x[0,0,1], ..., x[0,0,4],   x[0,1,0], ..., x[0,1,4],   ...
  x[1,0,0], x[1,0,1], ..., x[1,0,4],   ... ]
```

- **`axis=0`** sums `C[0, j, k] + C[1, j, k]` for each `(j, k)`. Each pair of operands is **5 million elements apart** in memory -- but the access pattern walks through the 5 columns of each row contiguously. Pretty cache-friendly.
- **`axis=1`** sums all $10^6$ elements of dimension 1 for each `(i, k)`. The pattern jumps by $5$ doubles ($40$ bytes) between accesses -- non-contiguous, blowing the cache.

In [180]:
%timeit F.sum(axis=0)

117 ms ± 20.3 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


And the F-ordered counterpart for axis=1:

In [181]:
%timeit F.sum(axis=1)

89.3 ms ± 1.07 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


**For F-order: the pattern reverses.** `axis=0` is now slowest (~99 ms), `axis=1` faster (~73 ms). And **both are slower than the C-order versions**.

Why? F-order means the **first axis (2) varies fastest in memory**. The buffer layout is:

```
[ C[0,0,0], C[1,0,0],   C[0,1,0], C[1,1,0],   ...   C[0,j,0], C[1,j,0], ... ]
```

The advantage when summing along axis=0 should be cache-friendliness, but the *huge* dimension (1,000,000) varies *second*-fastest, leading to long strided accesses for the result accumulator. The penalty far outweighs any gain.

**Performance summary:**

| Operation | C-order | F-order | Winner |
|---|---|---|---|
| `arr.sum()` | 6.67 ms | 6.55 ms | tie |
| `arr.sum(axis=0)` | **25.95 ms** | 99.19 ms | C, ~3.8x faster |
| `arr.sum(axis=1)` | **33.11 ms** | 73.01 ms | C, ~2.2x faster |

**Conclusions:**

1. **C-order is the better default for ML workloads.** Most operations read along feature-vectors (innermost axis) or aggregate per-sample (axis = 0).
2. **F-order can help only when the algorithm is genuinely column-wise** -- e.g., LAPACK linear-algebra routines that came from Fortran. NumPy detects this and may set `order='F'` internally for `np.linalg` calls.
3. **Strided / transposed views are the silent killer.** `arr.T.sum(axis=0)` is mathematically identical to `arr.sum(axis=1)`, but if `arr` is C-order, `arr.T` is F-order and the strided access slows it down. Use `np.ascontiguousarray(arr)` to materialize a contiguous copy when this matters.

**ML rule of thumb:** if a NumPy operation feels mysteriously slow, check `arr.flags['C_CONTIGUOUS']`. If `False`, consider `np.ascontiguousarray(arr)` before the hot loop.

## Conclusion

NumPy is the single most important third-party library for numerical Python. The `ndarray` class delivers:

| Feature | Benefit |
|---|---|
| Homogeneous dtype | 4-5x memory savings over Python lists |
| Contiguous memory | Cache-friendly; near-bandwidth performance |
| Vectorized C kernels | 10-100x speedup over Python loops |
| Broadcasting | Concise notation for elementwise + per-axis ops |
| Universal functions | `np.exp`, `np.log`, ... applied across whole arrays |
| Multi-dim indexing | `arr[i, j, k]`, `arr[:, mask]`, `arr.reshape(-1, 1)` |

**Three cardinal principles** to carry into Chapter 5 (pandas) and beyond:

1. **Vectorize, don't loop.** Pure-Python loops over array elements are 50-100x slower than NumPy expressions. Reach for `np.where`, broadcasting, and ufuncs first; drop to a Python loop only if the algorithm is irreducibly sequential.

2. **Memory layout matters at scale.** For million-element arrays, C-order vs F-order can change runtime by 3x. Default to C-order; check contiguity (`arr.flags`) when performance disappoints.

3. **Choose dtypes deliberately.** `float64` is the safe default for CPU-side preprocessing. Cast to `float32` before GPU upload for 2x training speedup. Avoid `object` dtype -- it's a vectorization killer.

In **Chapter 5**, we will see how **pandas** layers a high-level "labelled-axis" API on top of NumPy arrays. Pandas inherits NumPy's broadcasting, ufunc support, and dtype system, while adding row/column labels, missing-value handling, and the powerful `groupby` operation. Mastering NumPy is therefore the prerequisite for mastering pandas -- everything we did here is one level beneath every `df.<op>` call you will write.